# 02 - Classical statistical analysis (independent reproduction of figures and tables)

**Input**: four CSVs in `data/` - `NE_baseline_D1.csv`, `NE_baseline_D3.csv`, `NE_baseline_D7.csv`
(99 patients each) and `NE_HC.csv` (17 healthy controls)
**Output**: Table 1-3 and sTable S1-S2 (to `outputs/tables/`), Figure 2A/2B/2C and Figures S2-S5
(to `outputs/figures/`, png + pdf)

- Cohort: 99 patients (Day-1 SIC 56 / NSIC 43), grouping column `SIC_D1`; D3/D7 use `SIC_D3` / `SIC_D7`
- The whole notebook runs in about 1 minute and needs no external files

**Final figures**: Figure 2 and Supplementary Figures S2-S5 are assembled at the end of this notebook
with the layout of the reference figures (canvas size, panel positions, panel letters, font sizes),
written to `figures/` as `.png`, `.pdf` and `.tif` (300 dpi).


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

# ---- locate the package root (contains data/ and outputs/) ----
ROOT = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'outputs')):
        break
    ROOT = os.path.dirname(ROOT)

if not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'outputs'))):
    raise RuntimeError(
        'Package root not found (should contain data/ and outputs/): current directory = %s.'
        'Start Jupyter inside NE_99_261022_EN, or cd into it first.' % ROOT)
DATA = os.path.join(ROOT, 'data')
OUT_F = os.path.join(ROOT, 'outputs', 'figures'); os.makedirs(OUT_F, exist_ok=True)
OUT_T = os.path.join(ROOT, 'outputs', 'tables'); os.makedirs(OUT_T, exist_ok=True)

d1 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D1.csv'), encoding='utf-8-sig')
d3 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D3.csv'), encoding='utf-8-sig')
d7 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D7.csv'), encoding='utf-8-sig')
hc = pd.read_csv(os.path.join(DATA, 'NE_HC.csv'), encoding='utf-8-sig')

# Day-1 analysis cohort
cohort = d1[d1['SIC_D1'].isin(['SIC', 'NSIC'])].copy()
print('D1', d1.shape, '| D3', d3.shape, '| D7', d7.shape, '| HC', hc.shape)
print('cohort: SIC %d / NSIC %d' % ((cohort['SIC_D1']=='SIC').sum(), (cohort['SIC_D1']=='NSIC').sum()))

# font fallback
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

## Tables - five basic tables matching the structure of `tables/`

This section produces five tables with the **structure and order** used in `tables/` (values come
from this notebook):

| Table | Content | Data rows |
|---|---|---|
| Table 1 | Baseline characteristics (8 sections) | 73 |
| Table 2 | Day-1 neutrophil markers (HC / SIC / NSIC) | 5 |
| Table 3 | Diagnostic performance (AUC / AUPRC / operating point) | 8 |
| sTable S1 | Longitudinal distributions (Day 1/3/7) | 5 |
| sTable S2 | Longitudinal mixed-effects models (LMM) | 5 |

- Computation, formatting, LaTeX rendering and Word updates are handled by `tables/gen_tables.py`;
  the notebook only calls and displays.
- Intermediate tables -> `outputs/tables/table_*.csv`; LaTeX/PDF and docx updates:
  `python tables/build_tables.py` (or `gt.build_all()`).
- DeLong tests from Table 3 are **not** printed in the table; they go to `outputs/tables/table_t3_delong.csv`.


In [ ]:
import sys
sys.path.insert(0, os.path.join(ROOT, 'tables'))
import gen_tables as gt          # single source of truth (compute + format + LaTeX + Word)

t1 = gt.table1()
gt.write_csv(t1)                 # -> outputs/tables/table_t1.csv
display(t1.df)

## Table 2 - Day-1 neutrophil markers (HC / SIC / NSIC)

Median (IQR); CD10- is expressed as a percentage (two decimals); P is the Mann-Whitney U test for SIC vs NSIC.


In [ ]:
t2 = gt.table2()
gt.write_csv(t2)
display(t2.df)

## Table 3 - Diagnostic performance (Day-1 SIC)

- AUC / AUPRC / operating-point metrics: bootstrap 95% CI (B = 2000, seed = 20260507); markers with
  AUC < 0.5 are direction-reversed first;
- Thresholds: Youden optimum on the full sample, fixed before resampling;
- **DeLong is not part of the formatted table** (paired tests are not recommended in a diagnostic table)
  and is written separately to `table_t3_delong.csv`;
- The `results` object is reused by Figure 2B (same bootstrap draws).


In [ ]:
# dependencies for Figures 2B/2C (previously in the Table 3 cell)
from scipy import stats
from sklearn.metrics import roc_curve, auc, precision_recall_curve

t3, results = gt.table3()
gt.write_csv(t3)
delong_csv, delong_df = gt.write_delong(results)     # DeLong to CSV only, not in the table

# reused by Figures 2B / 2C
BM3 = gt.BM
cohort3 = cohort.copy()
for _c, _n, _s in BM3:
    cohort3[_c] = pd.to_numeric(cohort3[_c], errors='coerce')

display(t3.df)
display(delong_df)

## sTable S1 - Longitudinal distributions (HC + Day 1/3/7 x SIC/NSIC)

Box plots (median/IQR); P from Mann-Whitney U between SIC and NSIC within the same time point.


In [ ]:
s1 = gt.stable_s1()
gt.write_csv(s1)
display(s1.df)

## sTable S2 - Longitudinal mixed-effects models (LMM)

Model: `marker ~ time_f * SIC_Group + (1 | PatientID)`, with the Day-1 SIC group fixed throughout;
P values from joint Wald tests (time 2 df, group 1 df, interaction 2 df).


In [ ]:
t_s2, dfl, lmm_res, overall_wald, BM_L = gt.stable_s2(return_models=True)
gt.write_csv(t_s2)
display(t_s2.df)

## Figure 2A - Day-1 violin plots (SIC vs NSIC, five markers)

Violin + box + individual points; two-sided Mann-Whitney U between groups, P values above the brackets.
This cell defines `draw_fig2a_panel()`, reused by the composite Figure 2 below.


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

MARKERS = [b[0] for b in BM3]
LABEL2 = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
          'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
          'immature_Neu_percentage': 'CD10$^{-}$ neutrophils,%'}
SIC_EDGE, NSIC_EDGE = '#B2182B', '#1A4A8C'
SIC_FILL, NSIC_FILL = '#EBC5C9', '#C5D1E2'

# font sizes: SINGLE = standalone panel; COMPOSITE = composite (matched to the reference)
FS_2A_SINGLE = {'tick': 14, 'label': 16, 'p': 14}
FS_2A_COMPOSITE = {'tick': 17.5, 'label': 20.5, 'p': 19}

grp = cohort['SIC_D1']
rows, pvals = [], []
for m in MARKERS:
    v = pd.to_numeric(cohort[m], errors='coerce')
    a = v[(grp == 'SIC') & v.notna()]; b = v[(grp == 'NSIC') & v.notna()]
    p = float(mannwhitneyu(a, b, alternative='two-sided')[1])
    pvals.append(p)
    rows.append({'Marker': LABEL2[m], 'N SIC': len(a), 'N NSIC': len(b),
                 'Median SIC': round(float(a.median()), 4), 'Median NSIC': round(float(b.median()), 4),
                 'P (Mann-Whitney)': p})
stat2a = pd.DataFrame(rows)
stat2a['q (BH-FDR)'] = multipletests(pvals, method='fdr_bh')[1]
stat2a.to_csv(os.path.join(OUT_T, 'violin_D1_group_comparison.csv'), index=False, encoding='utf-8-sig')


def draw_fig2a_panel(ax, m, fs=None):
    """Draw one marker's SIC vs NSIC violin (+ box + points + significance bracket) on the given axes."""
    fs = dict(FS_2A_SINGLE, **(fs or {}))
    v = pd.to_numeric(cohort[m], errors='coerce')
    data = [v[(grp == 'SIC') & v.notna()].astype(float).values,
            v[(grp == 'NSIC') & v.notna()].astype(float).values]
    is_frac = 'immature' in m
    if is_frac:
        data = [arr * 100.0 for arr in data]
    rng42 = np.random.default_rng(42)
    vp = ax.violinplot(data, positions=[0, 1], widths=0.55, showmeans=False,
                       showmedians=False, showextrema=False)
    for body, fc, ec in zip(vp['bodies'], [SIC_FILL, NSIC_FILL], [SIC_EDGE, NSIC_EDGE]):
        body.set_facecolor(fc); body.set_alpha(1.0); body.set_edgecolor(ec); body.set_linewidth(1.4)
    bp = ax.boxplot(data, positions=[0, 1], widths=0.15, patch_artist=True, showfliers=False,
                    boxprops=dict(linewidth=1.3), whiskerprops=dict(color='#555555', linewidth=1.1),
                    capprops=dict(color='#555555', linewidth=1.1), medianprops=dict(linewidth=2.4))
    for box, med, ec, fc in zip(bp['boxes'], bp['medians'], [SIC_EDGE, NSIC_EDGE],
                                [SIC_FILL, NSIC_FILL]):
        box.set_facecolor(fc); box.set_edgecolor(ec); med.set_color(ec)
    for i, (arr, ec) in enumerate(zip(data, [SIC_EDGE, NSIC_EDGE])):
        ax.scatter(i + rng42.uniform(-0.075, 0.075, size=len(arr)), arr, s=14, color=ec,
                   alpha=0.45, edgecolors='none', zorder=3)
    lo = float(min(data[0].min(), data[1].min())); hi = float(max(data[0].max(), data[1].max()))
    span = hi - lo
    ax.set_ylim(lo - span * 0.06, hi + span * 0.26)
    p = float(mannwhitneyu(data[0], data[1], alternative='two-sided')[1])
    ptxt = 'P < 0.001' if p < 0.001 else 'P = %.3f' % p
    y_bar = hi + span * 0.10
    ax.plot([-0.02, 1.02], [y_bar, y_bar], color='black', linewidth=1.2,
            solid_capstyle='butt', zorder=6, clip_on=False)
    ax.text(0.5, y_bar + span * 0.015, ptxt, ha='center', va='bottom',
            fontsize=fs['p'], color='black', zorder=6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['SIC', 'NSIC'], fontsize=fs['tick'], fontweight='bold')
    ax.tick_params(axis='y', labelsize=fs['tick'])
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)' if is_frac else LABEL2[m],
                  fontsize=fs['label'], fontweight='bold', labelpad=0)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3); ax.set_axisbelow(True)


# ---- standalone panel export (outputs/figures) ----
fig, axes = plt.subplots(1, 5, figsize=(20, 4.6))
for ax, m in zip(axes, MARKERS):
    draw_fig2a_panel(ax, m)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'fig2A_violin_D1_5biomarkers.' + ext), dpi=300, bbox_inches='tight')
plt.show()
print(stat2a.to_string(index=False))

## Figure 2B - ROC curves (five markers, Day 1)

Curves and legend AUROCs (95% CI) are taken directly from Table 3 (same bootstrap draws).
This cell defines `draw_fig2b_panel()`.


In [ ]:
ROC_STYLE = [('CD63', 'CD63 MFI', '#B2182B'), ('CD62L', 'CD62L MFI', '#990099'),
             ('CD66b', 'CD66b MFI', '#339C62'), ('CD16', 'CD16 MFI', '#E69F00'),
             ('iNeu', 'CD10$^{-}$ neutrophils (%)', '#1A4A8C')]
col_of = dict([(b[2], b[0]) for b in BM3])
FS_2B_SINGLE = {'tick': 10, 'label': 14, 'legend': 9.5, 'annot': 11}
FS_2B_COMPOSITE = {'tick': 23, 'label': 23.5, 'legend': 16, 'annot': 17}


def draw_fig2b_panel(ax, fs=None):
    """Draw the five-marker Day-1 ROC (AUROCs from the same bootstrap draws as Table 3)."""
    fs = dict(FS_2B_SINGLE, **(fs or {}))
    ax.plot([0, 1], [0, 1], color='#8A8A8A', lw=1.5, ls='--', label='No-discrimination line')
    for short, lab, color in ROC_STYLE:
        r = results[short]
        d = cohort3[[col_of[short], 'SIC_D1']].dropna()
        xv = d[col_of[short]].values.astype(float)
        yv = (d['SIC_D1'].values == 'SIC').astype(int)
        fpr, tpr, _ = roc_curve(yv, -xv if r['flipped'] else xv)
        lo, hi = r['ci']['AUC']
        ax.plot(fpr, tpr, color=color, lw=2.5,
                label='%s: %.3f (%.3f\u2013%.3f)' % (lab, r['AUC'], lo, hi))
    r = results['iNeu']
    ax.text(0.97, 0.42, 'CD10$^{-}$ neutrophil cutoff = 96.5%%\nSensitivity %.3f\nSpecificity %.3f'
            % (r['Sens'], r['Spec']), transform=ax.transAxes, ha='right', va='top',
            fontsize=fs['annot'], style='italic', color='#222222')
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=fs['tick'])
    ax.tick_params(axis='y', labelsize=fs['tick'])
    ax.set_xlabel('1 - Specificity', fontsize=fs['label'], fontweight='bold')
    ax.set_ylabel('Sensitivity', fontsize=fs['label'], fontweight='bold')
    ax.legend(loc='lower right', fontsize=fs['legend'] * 0.9, frameon=True, fancybox=True,
              handlelength=1.5, labelspacing=0.32,
              facecolor='white', edgecolor='gray', borderpad=0.5)
    ax.grid(True, alpha=0.2, linestyle=':')


fig, ax = plt.subplots(figsize=(7.6, 6.4))
draw_fig2b_panel(ax)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'fig2B_roc_5biomarkers_day1.' + ext), dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

## Figure 2C - Restricted cubic spline (RCS) of CD10- and SIC risk

- RCS (4 knots: P5/P35/P65/P95), reference P10; overall association = RCS vs intercept LRT,
  non-linearity = RCS vs linear LRT
- Change point: piecewise-linear (hinge) logistic model, maximum likelihood on a 400-point P10-P90 grid
- This cell defines `draw_fig2c_panel()`


In [ ]:
import statsmodels.api as sm

x_raw = pd.to_numeric(cohort['immature_Neu_percentage'], errors='coerce')
mask = x_raw.notna() & cohort['SIC_D1'].isin(['SIC', 'NSIC'])
x3 = x_raw[mask].values.astype(float) * 100.0     # percentage scale
y3 = (cohort.loc[mask, 'SIC_D1'].values == 'SIC').astype(int)


def rcs_basis(xv, knots):
    k = np.sort(np.asarray(knots, float)); K = len(k)
    cols = [xv.astype(float).copy()]
    for j in range(0, K - 2):
        tj = k[j]
        t1 = np.clip(xv - tj, 0, None) ** 3
        t2 = np.clip(xv - k[K-2], 0, None) ** 3 * ((k[K-1] - tj) / (k[K-1] - k[K-2]))
        t3 = np.clip(xv - k[K-1], 0, None) ** 3 * ((k[K-2] - tj) / (k[K-1] - k[K-2]))
        cols.append(t1 - t2 + t3)
    return np.column_stack(cols)


knots4 = np.unique(np.percentile(x3, [5, 35, 65, 95]))
m_rcs = sm.GLM(y3, sm.add_constant(rcs_basis(x3, knots4)), family=sm.families.Binomial()).fit()
m_lin = sm.GLM(y3, sm.add_constant(x3), family=sm.families.Binomial()).fit()
m_nul = sm.GLM(y3, np.ones((len(x3), 1)), family=sm.families.Binomial()).fit()
p_glob = stats.chi2.sf(2 * (m_rcs.llf - m_nul.llf), df=len(knots4) - 1)
p_nl   = stats.chi2.sf(2 * (m_rcs.llf - m_lin.llf), df=len(knots4) - 2)
ref = float(np.percentile(x3, 10))


def or_vs_ref(model, v, ref):
    dv = np.column_stack([np.ones(1), rcs_basis(np.array([v]), knots4)])
    dr = np.column_stack([np.ones(1), rcs_basis(np.array([ref]), knots4)])
    d = dv - dr
    return float((dv @ model.params)[0] - (dr @ model.params)[0]), float(np.sqrt((d @ model.cov_params() @ d.T)[0, 0]))


# change point (piecewise-linear logistic, grid maximum likelihood)
c_grid = np.linspace(float(np.percentile(x3, 10)), float(np.percentile(x3, 90)), 400)
ll_grid = np.array([sm.GLM(y3, sm.add_constant(np.column_stack([x3, np.clip(x3 - c, 0, None)])),
                           family=sm.families.Binomial()).fit().llf for c in c_grid])
c_hat = float(c_grid[int(np.argmax(ll_grid))])
# ---- bootstrap 95% CI for the change point (patient-level, B=500; 100-point grid) ----
B_RCS = 500
rng_rcs = np.random.default_rng(42)
boot_c = []
for _b in range(B_RCS):
    ii = rng_rcs.integers(0, len(x3), len(x3))
    if len(np.unique(y3[ii])) < 2:
        continue
    xb, yb = x3[ii], y3[ii]
    grid_b = np.linspace(float(np.percentile(xb, 10)), float(np.percentile(xb, 90)), 100)
    lls = []
    for _c in grid_b:
        try:
            lls.append(sm.GLM(yb, sm.add_constant(np.column_stack([xb, np.clip(xb - _c, 0, None)])),
                              family=sm.families.Binomial()).fit().llf)
        except Exception:
            lls.append(-np.inf)
    boot_c.append(grid_b[int(np.argmax(lls))])
boot_c = np.asarray(boot_c, float)
c_lo, c_hi = np.percentile(boot_c, [2.5, 97.5])
print('Overall P = %.4g | Non-linearity P = %.4g | exploratory change point = %.1f%% '
      '(bootstrap 95%% CI %.1f-%.1f, B=%d)' % (p_glob, p_nl, c_hat, c_lo, c_hi, B_RCS))

xg = np.linspace(float(np.percentile(x3, 2)), float(np.percentile(x3, 98)), 400)
lor = np.array([or_vs_ref(m_rcs, v, ref) for v in xg])
or_g = np.exp(lor[:, 0]); or_lo = np.exp(lor[:, 0] - 1.96 * lor[:, 1]); or_hi = np.exp(lor[:, 0] + 1.96 * lor[:, 1])
or_c_pt = float(np.exp(or_vs_ref(m_rcs, c_hat, ref)[0]))

FS_2C_SINGLE = {'tick': 12, 'label': 14, 'annot': 12}
FS_2C_COMPOSITE = {'tick': 21.5, 'label': 24, 'annot': 18}


def draw_fig2c_panel(ax, fs=None):
    """Draw the RCS of CD10- against SIC risk (OR curve + 95% CI + change point)."""
    fs = dict(FS_2C_SINGLE, **(fs or {}))
    MAIN, RED = '#1A4A8C', '#B2182B'
    ax.fill_between(xg, or_lo, or_hi, color=MAIN, alpha=0.15, lw=0)
    ax.plot(xg, or_g, color=MAIN, lw=2.5)
    ax.axhline(1.0, color=RED, ls='--', lw=1.2)
    ax.text(xg.min() + 1, 2.5, 'OR = 1', fontsize=fs['annot'], color='#222222', style='italic')
    ax.axvline(c_hat, color=RED, ls='--', lw=1.8)
    ax.plot([c_hat], [or_c_pt], marker='o', ms=11, mfc='white', mec=RED, mew=2.2, zorder=6)
    ax.annotate('Exploratory change point = %.1f%%\n(bootstrap 95%% CI %.1f\u2013%.1f)'
                % (c_hat, c_lo, c_hi), xy=(c_hat, or_c_pt),
                xytext=(0.42, 0.26), textcoords='axes fraction', fontsize=fs['annot'], color=RED,
                fontweight='bold', ha='center', va='center',
                arrowprops=dict(arrowstyle='->', color=RED, lw=1.4, shrinkA=3, shrinkB=8))
    t1_ = 'Overall association: ' + ('$P$ < 0.001' if p_glob < 0.001 else '$P$ = %.3f' % p_glob)
    t2_ = 'Non-linearity: ' + ('$P$ < 0.001' if p_nl < 0.001 else '$P$ = %.3f' % p_nl)
    ax.text(0.03, 0.97, t1_ + '\n' + t2_, transform=ax.transAxes, va='top', ha='left',
            fontsize=fs['annot'], fontweight='bold', color='#222222')
    ax.set_xlim(xg.min(), xg.max()); ax.set_ylim(0, 150)
    ax.set_xlabel('CD10$^{-}$ neutrophils (%)', fontsize=fs['label'], fontweight='bold')
    ax.set_ylabel('Odds ratio for concurrent SIC', fontsize=fs['label'], fontweight='bold')
    ax.tick_params(labelsize=fs['tick'])
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)


fig, ax = plt.subplots(figsize=(7.6, 6.4))
draw_fig2c_panel(ax)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'fig2C_rcs_imm_neu.' + ext), dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

## Figure S2 - Longitudinal distributions (HC + D1/D3/D7 x SIC/NSIC)

Box plots + points; brackets show Mann-Whitney tests: black = SIC vs NSIC (same day), red = SIC vs HC,
blue = NSIC vs HC (drawn only when P < 0.05).
This cell defines `draw_s2_panel()` / `s2_legend()`.


In [ ]:
from matplotlib.gridspec import GridSpec

FIG_LABEL_S2 = {'All_Neu(CD63_MFI)': 'CD63 MFI in total neutrophils',
                'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI in total neutrophils',
                'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI in total neutrophils',
                'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI in total neutrophils',
                'immature_Neu_percentage': 'CD10$^{-}$ neutrophils percentage'}
colors_s2 = {'SIC': '#FF9999', 'NSIC': '#99CCFF', 'HC': '#CCFF99'}
box_colors_s2 = {'SIC': '#B2182B', 'NSIC': '#1A4A8C', 'HC': '#339C62'}
markers_s2 = {'SIC': 'o', 'NSIC': '^', 'HC': 's'}
GROUPS = ['HC', 'Day1_NSIC', 'Day1_SIC', 'Day3_NSIC', 'Day3_SIC', 'Day7_NSIC', 'Day7_SIC']
XLAB = ['HC', 'D1 NSIC', 'D1 SIC', 'D3 NSIC', 'D3 SIC', 'D7 NSIC', 'D7 SIC']
FS_S2_SINGLE = {'tick': 14, 'label': 14, 'title': 16, 'star': 12}
FS_S2_COMPOSITE = {'tick': 15, 'label': 17.5, 'title': 18.5, 'star': 12}


def group_vals_s2(m):
    g = {}
    for name, dd, gc in [('Day1', d1, 'SIC_D1'), ('Day3', d3, 'SIC_D3'), ('Day7', d7, 'SIC_D7')]:
        for s_ in ['SIC', 'NSIC']:
            g[name + '_' + s_] = pd.to_numeric(dd.loc[dd[gc] == s_, m], errors='coerce').dropna().values
    g['HC'] = pd.to_numeric(hc[m], errors='coerce').dropna().values
    return g


def draw_bracket(ax, x1, x2, y, off, stars, color, fs_star=12):
    ax.plot([x1, x1], [y - off, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.plot([x2, x2], [y - off, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.plot([x1, x2], [y, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.text((x1 + x2) / 2, y + off * 0.5, stars, ha='center', va='bottom',
            fontsize=fs_star, fontweight='bold', color=color)


def stars_of(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else None


def draw_s2_panel(ax, m, fs=None):
    """Draw one marker's longitudinal distribution (HC + D1/D3/D7 x SIC/NSIC)."""
    fs = dict(FS_S2_SINGLE, **(fs or {}))
    gv = group_vals_s2(m)
    if 'immature' in m:
        gv = {k: v * 100.0 for k, v in gv.items()}
    x_base = {name: i + 1 for i, name in enumerate(GROUPS)}
    bdata = [gv[n] for n in GROUPS if len(gv[n])]
    bpos = [x_base[n] for n in GROUPS if len(gv[n])]
    bcol = [box_colors_s2['HC' if n == 'HC' else n.split('_')[1]] for n in GROUPS if len(gv[n])]
    bp = ax.boxplot(bdata, positions=bpos, widths=0.6, patch_artist=True, showfliers=False, zorder=5)
    for patch, c in zip(bp['boxes'], bcol):
        patch.set(facecolor=c, alpha=0.25, edgecolor=c, linewidth=1.5)
    for med, c in zip(bp['medians'], bcol):
        med.set(color=c, linewidth=2.5)
    for w in bp['whiskers']:
        w.set(color='black', linewidth=1.2)
    for c_ in bp['caps']:
        c_.set(color='black', linewidth=1.2)
    rng_s2 = np.random.default_rng(7)
    for n in GROUPS:
        if len(gv[n]) == 0:
            continue
        gname = 'HC' if n == 'HC' else n.split('_')[1]
        ax.scatter(x_base[n] + rng_s2.uniform(-0.15, 0.15, len(gv[n])), gv[n],
                   c=colors_s2[gname], marker=markers_s2[gname], s=48, alpha=0.85,
                   edgecolors='black', linewidth=0.6, zorder=10)
    all_v = np.concatenate([v for v in gv.values() if len(v)])
    y_min, y_max = np.nanmin(all_v), np.nanmax(all_v)
    if y_max > 1e6:
        y_max = np.nanpercentile(all_v, 95) * 1.2
    margin = (y_max - y_min) * 0.15 if y_max != y_min else max(abs(y_max) * 0.1, 0.1)
    y_low, y_high = y_min - margin, (y_max + margin) * 1.08
    ax.set_ylim(y_low, y_high)
    ax.set_xlim(0.2, 7.8)
    ax.set_xticks([x_base[n] for n in GROUPS])
    ax.set_xticklabels(XLAB, fontsize=fs['tick'], rotation=45, ha='right')
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)' if 'immature' in m else 'Mean fluorescence intensity',
                  fontsize=fs['label'])
    ax.tick_params(axis='y', labelsize=fs['tick'])
    yl = ax.get_ylim(); ys = yl[1] - yl[0]
    y_sh, y_nh, y_sn = yl[1] - ys * 0.0475, yl[1] - ys * 0.095, yl[1] - ys * 0.1425
    for day in ['Day1', 'Day3', 'Day7']:
        for comp, ypos, col in [('SIC', y_sh, '#B2182B'), ('NSIC', y_nh, '#1A4A8C')]:
            a_, b_ = gv[day + '_' + comp], gv['HC']
            if len(a_) and len(b_):
                p = mannwhitneyu(a_, b_, alternative='two-sided')[1]
                if p < 0.05:
                    draw_bracket(ax, x_base[day + '_' + comp], x_base['HC'], ypos, ys * 0.03,
                                 stars_of(p), col, fs['star'])
        a_, b_ = gv[day + '_SIC'], gv[day + '_NSIC']
        if len(a_) and len(b_):
            p = mannwhitneyu(a_, b_, alternative='two-sided')[1]
            if p < 0.05:
                draw_bracket(ax, x_base[day + '_SIC'], x_base[day + '_NSIC'], y_sn, ys * 0.03,
                             stars_of(p), 'black', fs['star'])
    ax.set_ylim(yl[0], max(yl[1], y_nh + ys * 0.05) * 1.02)
    ax.set_title(FIG_LABEL_S2[m], fontsize=fs['title'], fontweight='bold', pad=14)
    ax.grid(True, alpha=0.2, linestyle=':', axis='y')


def s2_legend(ax_leg, size):
    ax_leg.axis('off')
    ax_leg.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors_s2['SIC'], markersize=13, label='SIC'),
        plt.Line2D([0], [0], marker='^', color='w', markerfacecolor=colors_s2['NSIC'], markersize=13, label='NSIC'),
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=colors_s2['HC'], markersize=13, label='HC'),
        plt.Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='SIC vs. NSIC'),
        plt.Line2D([0], [0], color='#B2182B', lw=2.5, linestyle='--', label='SIC vs. HC'),
        plt.Line2D([0], [0], color='#1A4A8C', lw=2.5, linestyle='--', label='NSIC vs. HC')],
        loc='center', prop={'size': size, 'weight': 'bold'}, frameon=True, fancybox=True,
        facecolor='white', edgecolor='gray', borderpad=1.2)


# ---- standalone panel export (outputs/figures) ----
fig = plt.figure(figsize=(24, 16))
gs = GridSpec(2, 3, figure=fig, left=0.06, right=0.94, top=0.95, bottom=0.08, wspace=0.30, hspace=0.25)
for idx, m in enumerate(MARKERS):
    draw_s2_panel(fig.add_subplot(gs[idx // 3, idx % 3]), m)
s2_legend(fig.add_subplot(gs[1, 2]), 16)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'figS2_longitudinal_distributions.' + ext), dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

## Figure S3 - Longitudinal trajectories (individual lines + mean +/- SEM)

Grouped by Day-1 status (SIC/NSIC); each panel shows the overall LMM interaction P (same model as sTable S2).
This cell defines `draw_s3_panel()` / `s3_legend()`.


In [ ]:
from matplotlib.lines import Line2D

FIG_LABEL_S3 = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
                'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
                'immature_Neu_percentage': 'CD10$^{-}$ neutrophils (%)'}
p_inter = {m: overall_wald(lmm_res[m], 'interaction')[1] for m in BM_L}
line_c = {'SIC': '#FF9999', 'NSIC': '#99CCFF'}
mean_c = {'SIC': '#B2182B', 'NSIC': '#1A4A8C'}
FS_S3_SINGLE = {'tick': 14, 'label': 16, 'annot': 11}
FS_S3_COMPOSITE = {'tick': 14, 'label': 17.5, 'annot': 12}


def draw_s3_panel(ax, m, fs=None):
    """Draw one marker's trajectory (individual lines + mean +/- SEM + interaction P)."""
    fs = dict(FS_S3_SINGLE, **(fs or {}))
    plot_df = dfl[['PatientID', 'time', 'SIC_Group', m]].dropna().copy()
    if 'immature' in m:
        plot_df[m] = plot_df[m] * 100.0
    plot_df['Group'] = plot_df['SIC_Group'].map({0: 'NSIC', 1: 'SIC'})
    for pid in plot_df['PatientID'].unique():
        one = plot_df[plot_df['PatientID'] == pid].sort_values('time')
        if len(one) < 2:
            continue
        ax.plot(one['time'], one[m], color=line_c[one['Group'].iloc[0]], alpha=0.3,
                linewidth=1, solid_capstyle='round')
    st = plot_df.groupby(['time', 'Group'])[m].agg(['mean', 'sem']).reset_index()
    for gname in ['SIC', 'NSIC']:
        dd = st[st['Group'] == gname]
        ax.plot(dd['time'], dd['mean'], color=mean_c[gname], linewidth=3, marker='o', markersize=8)
        ax.fill_between(dd['time'], dd['mean'] - dd['sem'], dd['mean'] + dd['sem'],
                        color=mean_c[gname], alpha=0.2)
    pv = p_inter[m]
    ptxt = 'P < 0.001' if pv < 0.001 else 'P = %.3f' % pv
    ax.text(0.97, 0.96, 'Overall interaction ' + ptxt, transform=ax.transAxes, ha='right', va='top',
            fontsize=fs['annot'], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))
    ax.set_ylabel(FIG_LABEL_S3[m], fontsize=fs['label'], fontweight='bold')
    ax.tick_params(axis='both', labelsize=fs['tick'])
    ax.grid(True, alpha=0.3)
    ax.set_xticks([1, 3, 7]); ax.set_xticklabels(['Day 1', 'Day 3', 'Day 7'], fontsize=fs['tick'])


def s3_legend(ax_leg, size, title_size=None):
    ax_leg.clear(); ax_leg.axis('off')
    ax_leg.legend(handles=[
        Line2D([0], [0], color=mean_c['SIC'], linewidth=3, label='Day 1 SIC'),
        Line2D([0], [0], color=mean_c['NSIC'], linewidth=3, label='Day 1 NSIC')],
        loc='center', prop={'size': size, 'weight': 'bold'},
        title='SIC group', title_fontsize=title_size or size + 1.5, frameon=True,
        edgecolor='gray', facecolor='white')


# ---- standalone panel export (outputs/figures) ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()
for i, m in enumerate(BM_L):
    draw_s3_panel(axes_flat[i], m)
axes_flat[5].clear(); axes_flat[5].axis('off')
s3_legend(axes_flat[5], 14, 15)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'figS3_longitudinal_trajectories.' + ext), dpi=300, bbox_inches='tight')
plt.show()

## Figure S4 - Spearman heatmap: Day-1 markers x clinical variables

- Test family = 5 markers x 31 clinical variables = 155 Spearman tests, BH-FDR adjusted jointly
- Cells show rho; asterisks follow q values: \* q<0.05, \*\* q<0.01, \*\*\* q<0.001
- This cell defines `draw_s4_heatmap()`


In [ ]:
from scipy.stats import spearmanr
import seaborn as sns

# 31 clinical variables (column -> display name); fixed order matching the final figure
CLIN_GROUPS = [
    # 1 disease-severity scores
    [('SIC_score', 'SIC score'), ('DIC', 'DIC score'), ('APACHEII', 'APACHEII'), ('SOFA', 'SOFA')],
    # 2 coagulation
    [('PLT', 'PLT'), ('PT', 'PT'), ('APTT', 'APTT'), ('FIB', 'FIB'), ('INR', 'INR'),
     ('FDP', 'FDP'), ('D_dimer', 'D-dimer'), ('PTA', 'PTA'), ('ATIII', 'ATIII'), ('TT', 'TT')],
    # 3 blood count and inflammation
    [('WBC', 'WBC'), ('NE count', 'ANC'), ('LY count', 'ALC'), ('MO count', 'AMC'), ('NE/LY', 'NLR')],
    # 4 biochemistry and metabolism
    [('pH', 'pH'), ('HCO3_', 'Bicarbonate'), ('Ca2+', 'Ca$^{2+}$'), ('lac', 'Lactate')],
    # 5 SOFA organ subscores
    [('SOFA D1 Respiratory System', 'SOFA-Resp'), ('SOFA D1 Platelets', 'SOFA-Coag'),
     ('SOFA D1 Circulation', 'SOFA-CV'), ('SOFA D1 GCS Score', 'GCS'),
     ('SOFA D1 Creatinine', 'Creatinine'), ('SOFA D1 Bilirubin', 'TBIL')],
    # 6 treatments
    [('CRRT_D1', 'CRRT'), ('VT_D1', 'MV')],
]
CLIN32 = [item for grp in CLIN_GROUPS for item in grp]
GROUP_BOUNDS = list(np.cumsum([len(g) for g in CLIN_GROUPS])[:-1])
ROW_LABEL = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
             'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
             'immature_Neu_percentage': 'CD10$^{-}$ neutrophils (%)'}

corr_results = []
for m in MARKERS:
    for c, _disp in CLIN32:
        d = cohort[[m, c]].apply(pd.to_numeric, errors='coerce').dropna()
        r_, p_ = spearmanr(d[m], d[c]) if len(d) >= 3 else (np.nan, np.nan)
        corr_results.append({'Biomarker': m, 'Clinical Variable': c, 'Spearman rho': r_,
                             'p_value': p_, 'n': len(d)})
results_df = pd.DataFrame(corr_results)
valid = results_df['p_value'].notna()
results_df.loc[valid, 'q_value_BH'] = multipletests(results_df.loc[valid, 'p_value'].values, method='fdr_bh')[1]
results_df.to_csv(os.path.join(OUT_T, 'correlation_D1_spearman_detail.csv'), index=False, encoding='utf-8-sig')

corr_mat = results_df.pivot(index='Biomarker', columns='Clinical Variable', values='Spearman rho').reindex(
    index=MARKERS, columns=[c for c, _ in CLIN32]).astype(float)
qval_mat = results_df.pivot(index='Biomarker', columns='Clinical Variable', values='q_value_BH').reindex(
    index=MARKERS, columns=[c for c, _ in CLIN32]).astype(float)
corr_mat.index = [ROW_LABEL[m] for m in MARKERS]; corr_mat.columns = [d_ for _, d_ in CLIN32]
qval_mat.index = corr_mat.index; qval_mat.columns = corr_mat.columns


def annot_star(r, q):
    """Numbers on the first line, asterisks on the second (space when absent), keeping baselines aligned."""
    if np.isnan(r):
        return ''
    star = '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else ' '
    return '%.2f\n%s' % (r, star)


annot = corr_mat.copy().astype(str)
for i in range(5):
    for j in range(len(CLIN32)):
        annot.iloc[i, j] = annot_star(corr_mat.iloc[i, j], qval_mat.iloc[i, j])

FS_S4_SINGLE = {'annot': 11, 'tick': 13, 'xlab': 16, 'ylab': 15, 'cbar': 11}
FS_S4_COMPOSITE = {'annot': 11.5, 'tick': 14, 'xlab': 17, 'ylab': 16, 'cbar': 12.5,
                   'lw': 0.0}


def draw_s4_heatmap(ax, cax=None, fs=None, square=True):
    """Spearman heatmap of 5 markers x 31 clinical variables (BH-FDR asterisks).

    Numbers are placed at the cell centre (seaborn annot not used) and asterisks shifted down,
    so every number sits on the same baseline.
    """
    fs = dict(FS_S4_SINGLE, **(fs or {}))
    kw = dict(annot=False, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
              square=square, linewidths=fs.get('lw', 0.5), ax=ax,
              cbar_kws={'label': 'Spearman \u03c1'})
    if cax is not None:
        kw['cbar_ax'] = cax
    sns.heatmap(corr_mat, **kw)

    # manual annotation: centre the number, drop the asterisk
    for i in range(corr_mat.shape[0]):
        for j in range(corr_mat.shape[1]):
            v = corr_mat.iloc[i, j]
            if np.isnan(v):
                continue
            q = qval_mat.iloc[i, j]
            star = '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else ''
            ax.text(j + 0.5, i + 0.5, '%.2f' % v, ha='center', va='center',
                    fontsize=fs['annot'], color='#222222')
            if star:
                ax.text(j + 0.5, i + 0.5 + 0.35, star, ha='center', va='center',
                        fontsize=fs['annot'] * 0.80, color='#222222')

    ax.tick_params(axis='x', rotation=90, labelsize=fs['tick'])
    ax.tick_params(axis='y', rotation=0, labelsize=fs['tick'])
    for lbl in ax.get_yticklabels():          # bold the five markers
        lbl.set_fontweight('bold')
    ax.set_xlabel('Clinical Variables', fontsize=fs['xlab'], fontweight='bold')
    ax.set_ylabel('Neutrophil Biomarkers', fontsize=fs['ylab'], fontweight='bold')
    if cax is not None:
        ticks = [-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.0]
        cax.set_yticks(ticks)
        cax.set_yticklabels(['%.2f' % t for t in ticks])
        cax.tick_params(labelsize=fs['cbar'])
        cax.yaxis.label.set_size(fs['cbar'])
        cax.yaxis.label.set_fontweight('bold')   # bold the Spearman rho label
        cax.yaxis.labelpad = 14


# ---- standalone panel export (outputs/figures) ----
fig, ax = plt.subplots(figsize=(17.9, 4.4))
draw_s4_heatmap(ax, None)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'figS4_correlation_heatmap.' + ext), dpi=300, bbox_inches='tight')
plt.show()
n_q05 = int((results_df['q_value_BH'] < 0.05).sum())
print('BH family = %d tests; q<0.05 in %d' % (int(valid.sum()), n_q05))

## Figure S5 - CD10- against six coagulation/clinical variables (scatter + fitted line)

Spearman rho and BH q are annotated in the corner (same correction as Figure S4); fitted lines are
truncated at y = 100%.
This cell defines `draw_s5_panel()`.


In [ ]:
from scipy.stats import linregress

bio = 'immature_Neu_percentage'
coag_vars = ['PLT', 'lac', 'PT', 'ATIII', 'SIC_score', 'DIC']
coag_colors = ['#B2182B', '#1A4A8C', '#339C62', '#E69F00', '#990099', '#00A6A6']
coag_disp = {'PLT': 'Platelet count (\u00d710\u2079/L)', 'lac': 'Lactate (mmol/L)', 'PT': 'PT (s)',
             'ATIII': 'ATIII (%)', 'SIC_score': 'SIC score', 'DIC': 'DIC score'}
q_lookup = results_df.set_index(['Biomarker', 'Clinical Variable'])['q_value_BH']
FS_S5_SINGLE = {'tick': 14, 'title': 16, 'label': 16, 'annot': 12}
FS_S5_COMPOSITE = {'tick': 14.2, 'title': 16.5, 'label': 15, 'annot': 12.5}


def draw_s5_panel(ax, clin, fs=None):
    """Draw one CD10- vs coagulation/clinical scatter with a fitted line."""
    fs = dict(FS_S5_SINGLE, **(fs or {}))
    color = coag_colors[coag_vars.index(clin)]
    d = cohort[[bio, clin]].apply(pd.to_numeric, errors='coerce').dropna()
    xc = d[clin].values.astype(float); yc = d[bio].values.astype(float)
    r_, _ = spearmanr(xc, yc)
    q = q_lookup.get((bio, clin), np.nan)
    ax.scatter(xc, yc, c=color, s=40, alpha=0.55, edgecolors='white', linewidth=0.4)
    slope, intercept, _, _, _ = linregress(xc, yc)
    x_lo, x_hi = float(xc.min()), float(xc.max())
    if slope > 0:
        x_hi = min(x_hi, (1.0 - intercept) / slope)
    elif slope < 0:
        x_lo = max(x_lo, (1.0 - intercept) / slope)
    xl = np.linspace(x_lo, x_hi, 100)
    ax.plot(xl, slope * xl + intercept, color=color, linewidth=2.0, alpha=0.9)
    q_txt = 'q < 0.001' if (pd.notna(q) and q < 0.001) else ('q = %.3f' % q if pd.notna(q) else 'q = NA')
    ax.text(0.95, 0.05, '\u03c1 = %.2f\n%s' % (r_, q_txt), transform=ax.transAxes,
            fontsize=fs['annot'], ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
    ax.set_title(coag_disp[clin], fontsize=fs['title'], fontweight='bold')
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)', fontsize=fs['label'], fontweight='bold', labelpad=-8)
    ax.tick_params(labelsize=fs['tick'])
    ax.set_ylim(0.4, 1.0)
    ax.set_yticks([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_yticklabels(['40', '50', '60', '70', '80', '90', '100'])
    ax.grid(True, linestyle='--', alpha=0.3)


# ---- standalone panel export (outputs/figures) ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, clin in enumerate(coag_vars):
    draw_s5_panel(axes.flatten()[i], clin)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_F, 'figS5_imm_neu_vs_coagulation.' + ext), dpi=300, bbox_inches='tight')
plt.show()

## Composite layout - final figures matching the reference layout

The cells above export **single panels** (`outputs/figures/`); the last two cells of this notebook
assemble them into final figures:

- Canvas size, panel positions/spacing, panel letters, legend placement and font sizes are fixed to
  the reference figures;
- Output to `figures/`: `Figure 2` and `Supplementary Figures S2-S5`, each as `.png`, `.pdf` and
  `.tif` (300 dpi).


### Figure 2 (A: five violins / B: ROC / C: RCS)


In [ ]:
# ============================================================
#  composite layout - Figure 2 (panels A/B/C)
#  canvas size, panel positions and font sizes matched to the reference Figure 2;
#  output -> figures/Figure 2.png / .pdf / .tif (300 dpi)
# ============================================================
FIG_NEW = os.path.join(ROOT, 'figures')
os.makedirs(FIG_NEW, exist_ok=True)


def _new_canvas(W, H):
    return plt.figure(figsize=(W / 300, H / 300))


def _axes_at(fig, rect, W, H):
    x0, y0, x1, y1 = rect
    return fig.add_axes([x0 / W, 1 - y1 / H, (x1 - x0) / W, (y1 - y0) / H])


def _panel_letter(fig, x, y, W, H, txt, size=29):
    fig.text(x / W, 1 - y / H, txt, fontsize=size, va='top', ha='left')


def _savefig_retry(fig, path, **kw):
    """Write to a temporary file then rename (avoids intermittent EINVAL when the target is locked)."""
    import time
    root, ext = os.path.splitext(path)
    tmp = root + '.tmp%d' % os.getpid() + ext
    fig.savefig(tmp, **kw)
    for _ in range(5):
        try:
            os.replace(tmp, path)
            return
        except OSError:
            time.sleep(0.4)
    raise


def save_composite(fig, stem):
    """Export png / pdf, then convert the png to a 300 dpi tif (lossless, uncompressed)."""
    from PIL import Image
    png = os.path.join(FIG_NEW, stem + '.png')
    _savefig_retry(fig, png, dpi=300, facecolor='white')
    _savefig_retry(fig, os.path.join(FIG_NEW, stem + '.pdf'), dpi=300, facecolor='white')
    tif = os.path.join(FIG_NEW, stem + '.tif')
    Image.open(png).convert('RGB').save(tif, dpi=(300, 300))
    print('written:', png, '|', tif)


W, H = 6815, 4259
fig = _new_canvas(W, H)
for rect, m in zip([(336, 83, 1319, 1329), (1757, 73, 2736, 1319), (3149, 83, 4113, 1309),
                    (4513, 73, 5491, 1319), (5832, 73, 6811, 1314)], MARKERS):
    draw_fig2a_panel(_axes_at(fig, rect, W, H), m, FS_2A_COMPOSITE)
draw_fig2b_panel(_axes_at(fig, (331, 1625, 3257, 3973), W, H), FS_2B_COMPOSITE)
draw_fig2c_panel(_axes_at(fig, (3884, 1670, 6810, 3977), W, H), FS_2C_COMPOSITE)
_panel_letter(fig, 30, 10, W, H, 'A')
_panel_letter(fig, 30, 1600, W, H, 'B')
_panel_letter(fig, 3560, 1600, W, H, 'C')
save_composite(fig, 'Figure 2')
plt.show()

### Supplementary Figure S2–S5

In [ ]:
# ============================================================
#  composite layout - Supplementary Figures S2-S5
#  layout (canvas size / panel coordinates / legend / letters) matches the final figures
#  output -> figures/Supplementary Figure S2-S5 .png / .pdf / .tif
# ============================================================

# ---------- S2: five longitudinal distribution panels + legend ----------
W, H = 6746, 4629
fig = _new_canvas(W, H)
for rect, m in zip([(406, 115, 2169, 1971), (2694, 115, 4457, 1971), (4982, 115, 6745, 1971),
                    (406, 2493, 2169, 4349), (2694, 2493, 4457, 4349)], MARKERS):
    draw_s2_panel(_axes_at(fig, rect, W, H), m, FS_S2_COMPOSITE)
s2_legend(_axes_at(fig, (5240, 3058, 5980, 3780), W, H), 18)
for lab, (xx, yy) in zip('ABCDE', [(37, 36), (2325, 36), (4613, 36), (37, 2414), (2325, 2414)]):
    _panel_letter(fig, xx, yy, W, H, lab)
save_composite(fig, 'Supplementary Figure S2')
plt.show()

# ---------- S3: five trajectory panels + legend ----------
W, H = 4705, 2626
fig = _new_canvas(W, H)
for rect, m in zip([(296, 42, 1511, 1186), (1881, 42, 3096, 1186), (3466, 42, 4681, 1186),
                    (296, 1388, 1511, 2532), (1881, 1388, 3096, 2532)], BM_L):
    draw_s3_panel(_axes_at(fig, rect, W, H), m, FS_S3_COMPOSITE)
s3_legend(_axes_at(fig, (3735, 1750, 4345, 2095), W, H), 18, 20)
for lab, (xx, yy) in zip('ABCDE', [(34, 8), (1620, 8), (3205, 8), (34, 1354), (1620, 1354)]):
    _panel_letter(fig, xx, yy, W, H, lab, size=26)
save_composite(fig, 'Supplementary Figure S3')
plt.show()

# ---------- S4: correlation heatmap ----------
W, H = 5293, 1135
fig = _new_canvas(W, H)
ax_h = _axes_at(fig, (780, 22, 4780, 700), W, H)
draw_s4_heatmap(ax_h, _axes_at(fig, (4862, 22, 4922, 1006), W, H), FS_S4_COMPOSITE, square=False)
ax_h.set_ylabel('')
ax_h.tick_params(axis='y', pad=10)
fig.text(58 / W, 0.648, 'Neutrophil Biomarkers', rotation=90, va='center', ha='center',
         fontsize=FS_S4_COMPOSITE['tick'] + 2, fontweight='bold')
save_composite(fig, 'Supplementary Figure S4')
plt.show()

# ---------- S5: six scatter panels ----------
W, H = 4827, 2674
fig = _new_canvas(W, H)
for rect, clin in zip([(171, 90, 1539, 1234), (1823, 90, 3190, 1234), (3458, 90, 4825, 1234),
                       (175, 1436, 1543, 2580), (1818, 1436, 3195, 2580), (3450, 1436, 4826, 2580)],
                      coag_vars):
    draw_s5_panel(_axes_at(fig, rect, W, H), clin, FS_S5_COMPOSITE)
save_composite(fig, 'Supplementary Figure S5')
plt.show()